# Why the Assignments Use This Format

These assignments are structured around modular classes and functions because that is the only reliable way to measure actual understanding.
AI‑generated submissions tend to produce large procedural scripts that solve the problem without showing any grasp of the underlying concepts.  They look complete, but they reveal nothing about how the student thinks.

By requiring students to pull relevant pieces from earlier notebooks, they must demonstrate:

- they recognize patterns
- they understand how those patterns fit together
- they can adapt code instead of copying it
- they can document the changes they make
- they can assemble a working solution from modular parts

This format forces engagement with the material.
Using the right pieces of code, not the whole notebook, not an AI dump, shows at least partial understanding of the concepts we practiced all quarter.  It also mirrors real development work: you rarely write everything from scratch, but you must know how to adapt what already exists.

The goal is simple:
show your thinking, not just your output.


# Preface — The Purpose of this Notebook

This notebook shows how to take an existing example from previous notebooks and adapt it to solve a completely different sample assignment.

We start with the CaesarCipher notebook.
It contains a complete class: private state, accessors, internal helpers, public behavior, and a usable `__repr__`.
The cipher logic itself is not important.
The structure is.

This notebook demonstrates how to:

- extract the reusable parts of the CaesarCipher class
- identify which parts belong to the “complete class” pattern
- replace the cipher behavior with a new behavior
- integrate ideas from the Bit & Byte notebook (streaming, variable‑length output)
- produce a new class that solves a different problem

The new problem is Morse code encoding.
We build a `MorseCodeEncoder` using the CaesarCipher structure.
The actual Streaming assignment writes the `MorseCodeDecoder`.

The goal is to show the step‑by‑step transformation from one notebook’s example to a new assignment.
Every assignment in this course follows this same process:
start from a pattern, adapt it, document the changes, and produce a complete class.


# Step 1 — Start From the Existing CaesarCipher Notebook

Before building anything new, we look at the existing CaesarCipher notebook.
The goal is to identify the parts of the class that are reusable and the parts that must be replaced.

The CaesarCipher example is useful because it already shows:

- a complete class structure
- private internal state
- accessors using `@property`
- internal helper methods
- public behavior methods
- a usable `__repr__`

These structural elements will be reused.
The cipher logic will not.

This step is about understanding the pattern we are adapting, not about the cipher itself.

In [ ]:
class CaesarCipher:
    """
    A complete, modular, robust, scalable, and encapsulated Caesar cipher.
    """

    def __init__(self, shift, sequence=None):
        if not isinstance(shift, int):
            raise TypeError("Shift must be an integer")

        if sequence is None:
            sequence = [chr(code) for code in range(ord('A'), ord('Z') + 1)]

        if not isinstance(sequence, (list, tuple)):
            raise TypeError("Sequence must be a list or tuple")

        if not all(isinstance(ch, str) and len(ch) == 1 for ch in sequence):
            raise ValueError("Sequence must contain single-character strings")

        self.__shift = shift
        self.__sequence = list(sequence)

    @property
    def shift(self):
        return self.__shift

    @property
    def sequence(self):
        return list(self.__sequence)

    @property
    def size(self):
        return len(self.__sequence)

    def __index_of(self, ch):
        if ch not in self.__sequence:
            raise ValueError(f"Character '{ch}' not found in sequence")
        return self.__sequence.index(ch)

    def __shift_char(self, ch, amount):
        if not isinstance(ch, str) or len(ch) != 1:
            raise ValueError("Character must be a single character")

        if ch in self.__sequence:
            idx = self.__index_of(ch)
            new_index = (idx + amount) % self.size
            return self.__sequence[new_index]

        return ch

    def __process_text(self, text, amount):
        if not isinstance(text, str):
            raise TypeError("Text must be a string")
        return ''.join(self.__shift_char(ch, amount) for ch in text)

    def encrypt(self, text):
        return self.__process_text(text, self.__shift)

    def decrypt(self, text):
        return self.__process_text(text, -self.__shift)

    def __repr__(self):
        return f"CaesarCipher(shift={self.__shift}, size={self.size})"


# Step 2 — Identify What We Keep and What We Replace

The CaesarCipher class has two kinds of parts: structural parts and behavioral parts.
For this assignment, we keep the structure and replace the behavior.

We keep the structural parts because they form the “complete class” pattern:

- private internal state
- accessors using `@property`
- internal helper methods
- public behavior methods
- a useful `__repr__`

These parts are reusable across assignments.

We replace the behavioral parts because they are specific to Caesar cipher:

- shifting characters
- positional indexing
- modulo arithmetic
- rotation of a sequence

Morse code does not use positional shifts.
It uses a mapping from characters to dot‑dash strings.
So the structure stays the same, and the internal logic changes.

In [ ]:
# Illustration of "keep vs replace" using the CaesarCipher structure

# This is the original structure we are adapting.
# We are NOT keeping the cipher logic. We are keeping the class pattern.

class CaesarCipher:
    def __init__(self, shift, sequence):
        self.__shift = shift              # keep: private state
        self.__sequence = list(sequence)  # keep: private state

    @property
    def shift(self):                      # keep: accessor pattern
        return self.__shift

    @property
    def sequence(self):                   # keep: accessor pattern
        return list(self.__sequence)

    def __shift_char(self, ch):           # replace: cipher-specific helper
        pass

    def encrypt(self, text):              # replace: cipher-specific behavior
        pass

    def __repr__(self):                   # keep: dunder pattern
        return f"CaesarCipher(shift={self.__shift})"


# This is the adapted structure for Morse encoding.
# Same pattern. Different behavior.

class MorseCodeEncoder:
    def __init__(self, mapping):
        self.__mapping = dict(mapping)    # keep: private state (different data)

    @property
    def mapping(self):                    # keep: accessor pattern
        return dict(self.__mapping)

    def __encode_char(self, ch):          # replace: new helper
        return self.__mapping[ch]

    def encode(self, text):               # replace: new public behavior
        return ' '.join(self.__encode_char(ch) for ch in text)

    def __repr__(self):                   # keep: dunder pattern
        return f"MorseCodeEncoder(symbols={len(self.__mapping)})"


# Step 3 — Connect This to the Bit & Byte Notebook

The CaesarCipher notebook shows how to build a complete class with fixed‑width behavior.
The Bit & Byte notebook shows how to work with variable‑length output and streaming.
Morse code uses variable‑length symbols, so both notebooks are needed.

The Bit & Byte notebook introduces:

- variable‑length encodings
- streaming output
- emit patterns
- accumulator patterns
- mapping tables

These ideas carry directly into Morse encoding.
Each character becomes a dot‑dash string of different length, and the encoder must emit each symbol in sequence.
This is the same pattern used in the streaming exercises.

The goal of this step is to show how the two notebooks combine:
CaesarCipher provides the class structure, and Bit & Byte provides the streaming behavior.

In [ ]:
# Small excerpt showing how CaesarCipher logic is replaced by Morse logic.

# Original CaesarCipher helper (behavior we REPLACE)
def __shift_char(self, ch, amount):
    idx = self.__sequence.index(ch)
    new_index = (idx + amount) % len(self.__sequence)
    return self.__sequence[new_index]


# New MorseCodeEncoder helper (behavior we INSERT)
def __encode_char(self, ch):
    return self.__mapping[ch]


# Step 4 — Show How Caesar Logic Is Replaced by Morse Logic

At this point we have identified the parts of the CaesarCipher class that carry over and the parts that must be replaced.
Now we look directly at the code and show how the replacement works.
This is not the full transformation yet.
It is a small, focused example that shows the exact kind of change we will make later when we build the complete MorseCodeEncoder class.

The CaesarCipher helper uses positional indexing and modulo arithmetic.
The Morse encoder uses direct lookup into a mapping table.
The structure stays the same.
The behavior changes.

This step is about seeing the replacement logic in isolation before we apply it to the full class.

# Step 5 — Preparing for the Full CaesarCipher → MorseCodeEncoder Transformation

Up to this point, we have only looked at small pieces of code and small structural comparisons.
The next major step in this notebook will be to take the actual CaesarCipher class and walk through how each part is adapted into the MorseCodeEncoder class.
This is the main purpose of the notebook: show the transformation, not just the final result.

The CaesarCipher class provides:

- a complete class layout
- private internal state
- accessors
- internal helpers
- public behavior
- a usable `__repr__`

These parts will be carried over.

The cipher logic will be replaced with:

- a Morse mapping table
- a direct lookup helper
- a streaming encoder
- variable‑length output

This step is simply marking the transition.
The next section will begin the actual construction, piece by piece, showing how the CaesarCipher notebook becomes the MorseCodeEncoder notebook.

# Step 6 — Full Working Code Replacement (CaesarCipher → MorseCodeEncoder)

Up to this point, we have looked at small excerpts and structural comparisons.
Now we show the complete replacement: how the CaesarCipher class structure is reused and how the cipher logic is removed and replaced with Morse encoding logic.

This is not the final MorseCodeEncoder class.
This is the *transformation demonstration*: a side‑by‑side replacement showing how the CaesarCipher notebook becomes the MorseCodeEncoder notebook.

The goal is to make the adaptation process visible and concrete.
Students should be able to see exactly what stays, what goes, and what changes.

In [ ]:
# ------------------------------------------------------------
# CaesarCipher (original structure)
# ------------------------------------------------------------

class CaesarCipher:
    def __init__(self, shift, sequence):
        self.__shift = shift
        self.__sequence = list(sequence)

    @property
    def shift(self):
        return self.__shift

    @property
    def sequence(self):
        return list(self.__sequence)

    def __shift_char(self, ch, amount):
        idx = self.__sequence.index(ch)
        new_index = (idx + amount) % len(self.__sequence)
        return self.__sequence[new_index]

    def encrypt(self, text):
        return ''.join(self.__shift_char(ch, self.__shift) for ch in text)

    def __repr__(self):
        return f"CaesarCipher(shift={self.__shift})"


# ------------------------------------------------------------
# MorseCodeEncoder (replacement structure)
# ------------------------------------------------------------

class MorseCodeEncoder:
    def __init__(self, mapping):
        # same pattern: private internal state
        self.__mapping = dict(mapping)

    @property
    def mapping(self):
        # same pattern: accessor returns a copy
        return dict(self.__mapping)

    def __encode_char(self, ch):
        # replacement logic: direct lookup instead of positional shift
        if ch not in self.__mapping:
            raise ValueError(f"Character '{ch}' not in Morse mapping")
        return self.__mapping[ch]

    def encode(self, text):
        # replacement logic: streaming variable-length output
        return ' '.join(self.__encode_char(ch) for ch in text)

    def __repr__(self):
        # same pattern: useful dunder
        return f"MorseCodeEncoder(symbols={len(self.__mapping)})"


# ------------------------------------------------------------
# Example mapping and usage
# ------------------------------------------------------------

MORSE = {
    'A': '.-', 'B': '-...', 'C': '-.-.',
    'D': '-..', 'E': '.',   'F': '..-.',
    'G': '--.', 'H': '....','I': '..',
    'J': '.---','K': '-.-', 'L': '.-..',
    'M': '--',  'N': '-.',  'O': '---',
    'P': '.--.','Q': '--.-','R': '.-.',
    'S': '...', 'T': '-',   'U': '..-',
    'V': '...-','W': '.--', 'X': '-..-',
    'Y': '-.--','Z': '--..',
    ' ': '/'
}

encoder = MorseCodeEncoder(MORSE)
print(encoder.encode("HELLO WORLD"))


# Step 7 — What the Full Replacement Code Is Doing

The previous code cell showed the complete structural replacement: how the CaesarCipher class becomes the MorseCodeEncoder class.
This is not just a new class.
It is a demonstration of how the original notebook’s pattern is reused and how the cipher logic is removed and replaced.

The code is doing three things:

- keeping the CaesarCipher structure
- removing the CaesarCipher behavior
- inserting Morse encoding behavior

The private state, accessors, internal helper pattern, public method pattern, and `__repr__` all come directly from the CaesarCipher notebook.
Only the internal logic changes.

The CaesarCipher helper uses positional indexing and modulo arithmetic.
The Morse encoder uses direct lookup and variable‑length streaming.
The structure stays the same.
The behavior changes.

This section sets up the next step, where we document the exact changes made during the transformation.

# Step 8 — Documenting the Changes Made in the Transformation

Now that we have a full working code replacement, we need to document the changes that were made when adapting the CaesarCipher class into the MorseCodeEncoder class.
The goal is to make the transformation explicit.
Students should be able to see exactly what stayed the same, what was removed, and what was added.

The CaesarCipher class provided the structure:

- private internal state
- accessors
- internal helpers
- public behavior
- a useful `__repr__`

These parts were kept.

The cipher logic was removed:

- positional indexing
- modulo arithmetic
- rotation of a sequence

These parts were replaced with Morse encoding logic:

- a mapping table
- direct lookup
- variable‑length streaming output

This section will list the changes clearly and directly to follow the adaptation process step by step.


In [ ]:
# ------------------------------------------------------------
# Example of required documentation comments during adaptation
# ------------------------------------------------------------

class MorseCodeEncoder:
    def __init__(self, mapping):
        # Adapted from CaesarCipher.__init__:
        # - kept the private internal state pattern
        # - replaced 'shift' and 'sequence' with a mapping table
        # - mapping comes from Bit & Byte (variable-length symbols)
        self.__mapping = dict(mapping)

    @property
    def mapping(self):
        # Adapted from CaesarCipher accessors:
        # - same pattern: return a copy of internal state
        # - changed name and purpose (mapping instead of sequence)
        return dict(self.__mapping)

    def __encode_char(self, ch):
        # Replaces CaesarCipher.__shift_char:
        # - removed positional indexing and modulo arithmetic
        # - inserted direct lookup into mapping table
        # - variable-length output comes from Bit & Byte streaming examples
        if ch not in self.__mapping:
            raise ValueError(f"Character '{ch}' not in Morse mapping")
        return self.__mapping[ch]

    def encode(self, text):
        # Replaces CaesarCipher.encrypt:
        # - removed character shifting
        # - inserted streaming output pattern (emit each symbol)
        # - symbols separated by spaces (same pattern used in Bit & Byte)
        return ' '.join(self.__encode_char(ch) for ch in text)

    def __repr__(self):
        # Adapted from CaesarCipher.__repr__:
        # - same pattern: simple, useful summary of internal state
        # - changed fields to match Morse encoder
        return f"MorseCodeEncoder(symbols={len(self.__mapping)})"


# Step 10 — Summary of the Transformation

This notebook demonstrated how to adapt the CaesarCipher class into a MorseCodeEncoder class by keeping the structural pattern and replacing the behavior.
The CaesarCipher notebook provided the complete class layout.
The Bit & Byte notebook provided the variable‑length streaming ideas needed for Morse code.

The main points:

- the class structure was kept
- the cipher logic was removed
- the Morse mapping and streaming logic were added
- private state, accessors, helpers, public methods, and `__repr__` stayed in place
- positional indexing and modulo arithmetic were replaced with direct lookup and variable‑length output

The final result is a working MorseCodeEncoder class that follows the same design pattern as the CaesarCipher class but solves a different problem.
Students can now use this pattern to build the decoder and future assignments.


# Closing Remarks

This example is intentionally simple.
It shows how one notebook’s complete class can be adapted into another notebook’s assignment by keeping the structure and replacing the behavior.
The SQLite exercise will not be this clean.
Students will have to pull pieces from several notebooks we used this quarter.
Some of those pieces will be full classes.
Others will be small excerpts, like the bit‑streaming helpers.
The process is the same: identify the pattern, adapt it, document the changes, and assemble the parts into a working solution.
The code should be structured as modular classes and functions, not a procedural script.